 # Whisper Accent Robustness — Model Performance Evaluation



 Run `eval_model_perf.py` on SLURM first to generate the CSVs this notebook reads.



 - **Scripted**     → 6 held-out test speakers (never seen during training)

 - **Spontaneous**  → suitcase corpus (OOD, all speakers)



 Metrics:

 - **WER**  — word error rate (primary ASR metric)

 - **PER**  — phoneme error rate via G2P; labelled "PER (G2P)" throughout

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_DIR = "results/model_perf_comparison"

# Keys must match {model_key} in CSV filenames; values are display labels
MODEL_KEYS = {
    "baseline":      "Zero-shot",
    "baseline_lora": "Naive LoRA FT",
    "ctc_aux":       "CTC Aux",
    "feat_aux":      "Feat Aux",
    "both_aux":      "CTC + Feat",
}
SPLITS = ["scripted", "spontaneous"]


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from jiwer import wer as jiwer_wer

pd.set_option("display.max_colwidth", 80)


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def load_results(model_key: str, split: str) -> pd.DataFrame | None:
    p = Path(RESULTS_DIR) / f"{model_key}_{split}_predictions.csv"
    if not p.exists():
        print(f"  [missing] {p} — run eval_model_perf.py first")
        return None
    return pd.read_csv(p)


def available(key: str, split: str) -> bool:
    return results.get(key, {}).get(split) is not None


def corpus_wer(df: pd.DataFrame) -> float:
    return float(jiwer_wer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))


def corpus_per(df: pd.DataFrame) -> float | None:
    """Mean utterance PER (G2P-derived), precomputed by eval_model_perf.py."""
    if "utt_per" not in df.columns:
        return None
    vals = df["utt_per"].dropna()
    return float(vals.mean()) if len(vals) else None


def grouped_wer(df: pd.DataFrame, group_col: str = "l1") -> pd.DataFrame:
    rows = []
    for grp, sub in df.groupby(group_col):
        rows.append({
            group_col:  grp,
            "num_utts": len(sub),
            "wer":      float(jiwer_wer(
                            sub["reference_norm"].fillna("").tolist(),
                            sub["prediction_norm"].fillna("").tolist(),
                        )),
            "per":      float(sub["utt_per"].dropna().mean())
                        if "utt_per" in sub.columns else None,
        })
    return pd.DataFrame(rows)


print("Helpers loaded.")


Helpers loaded.


In [4]:
# ── Load all cached CSVs ──────────────────────────────────────────────────────
results: dict[str, dict[str, pd.DataFrame | None]] = {}
for key in MODEL_KEYS:
    results[key] = {}
    for split in SPLITS:
        df = load_results(key, split)
        results[key][split] = df
        if df is not None:
            print(f"  loaded  {key}/{split}: {len(df):,} rows")


  loaded  baseline/scripted: 6,506 rows
  loaded  baseline/spontaneous: 22 rows
  loaded  baseline_lora/scripted: 6,506 rows
  loaded  baseline_lora/spontaneous: 22 rows
  loaded  ctc_aux/scripted: 6,506 rows
  loaded  ctc_aux/spontaneous: 22 rows
  [missing] results/model_perf_comparison/feat_aux_scripted_predictions.csv — run eval_model_perf.py first
  [missing] results/model_perf_comparison/feat_aux_spontaneous_predictions.csv — run eval_model_perf.py first
  [missing] results/model_perf_comparison/both_aux_scripted_predictions.csv — run eval_model_perf.py first
  [missing] results/model_perf_comparison/both_aux_spontaneous_predictions.csv — run eval_model_perf.py first


 ---

 # Part 1 — Overall WER & PER (G2P)

In [5]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for key, label in MODEL_KEYS.items():
    for split in SPLITS:
        if not available(key, split):
            continue
        df  = results[key][split]
        wer = corpus_wer(df)
        per = corpus_per(df)
        rows.append({"Model": label, "Split": split, "WER": wer, "PER (G2P)": per})

overall_df = pd.DataFrame(rows)

for metric in ["WER", "PER (G2P)"]:
    pivot = overall_df.pivot(index="Model", columns="Split", values=metric)
    display(
        pivot.style
             .format("{:.4f}")
             .background_gradient(cmap="RdYlGn_r", axis=None)
             .set_caption(f"{metric} by Model × Split (lower is better)")
    )


Split,scripted,spontaneous
Model,,
CTC Aux,18.4300,0.6297
Naive LoRA FT,0.0889,0.6261
Zero-shot,0.1841,0.6369


Split,scripted,spontaneous
Model,,
CTC Aux,16.8661,0.5378
Naive LoRA FT,0.0549,0.5366
Zero-shot,0.1004,0.5448


In [6]:
# ── Bar charts: WER and PER side-by-side, one figure per split ───────────────
for split in SPLITS:
    sub = overall_df[overall_df["Split"] == split].dropna(subset=["WER"])
    if sub.empty:
        continue

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name         = "WER",
        x            = sub["Model"].tolist(),
        y            = sub["WER"].tolist(),
        text         = [f"{v:.1%}" for v in sub["WER"]],
        textposition = "outside",
    ))
    if sub["PER (G2P)"].notna().any():
        fig.add_trace(go.Bar(
            name         = "PER (G2P)",
            x            = sub["Model"].tolist(),
            y            = sub["PER (G2P)"].tolist(),
            text         = [f"{v:.1%}" if pd.notna(v) else "" for v in sub["PER (G2P)"]],
            textposition = "outside",
        ))
    fig.update_layout(
        title   = f"WER & PER by Model — {split.capitalize()}",
        barmode = "group",
        yaxis   = dict(title="Error Rate", tickformat=".0%"),
        legend  = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin  = dict(t=80),
    )
    fig.show()


 ---

 # Part 2 — Per-L1 WER

In [7]:
for split in SPLITS:
    l1_rows = []
    for key, label in MODEL_KEYS.items():
        if not available(key, split):
            continue
        grp = grouped_wer(results[key][split], "l1")
        grp["Model"] = label
        l1_rows.append(grp)
    if not l1_rows:
        continue

    l1_df = pd.concat(l1_rows, ignore_index=True)

    # Delta vs zero-shot baseline (negative = improvement)
    base_label = MODEL_KEYS["baseline"]
    base_wer   = (
        l1_df[l1_df["Model"] == base_label][["l1", "wer"]]
        .rename(columns={"wer": "wer_base"})
    )
    l1_df = l1_df.merge(base_wer, on="l1", how="left")
    l1_df["wer_delta_pct"] = (
        (l1_df["wer"] - l1_df["wer_base"]) / l1_df["wer_base"] * 100
    )

    l1s = sorted(l1_df["l1"].unique())

    fig = go.Figure()
    for key, label in MODEL_KEYS.items():
        sub = l1_df[l1_df["Model"] == label].set_index("l1")
        fig.add_trace(go.Bar(
            name = label,
            x    = l1s,
            y    = [sub.loc[l, "wer"] if l in sub.index else None for l in l1s],
            text = [f"{sub.loc[l, 'wer']:.1%}" if l in sub.index else "" for l in l1s],
            textposition = "outside",
        ))
    fig.update_layout(
        title   = f"WER by L1 — {split.capitalize()}",
        barmode = "group",
        yaxis   = dict(title="WER", tickformat=".0%"),
        legend  = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin  = dict(t=80),
    )
    fig.show()

    out = Path(RESULTS_DIR) / f"comparison_{split}_by_l1.csv"
    l1_df.to_csv(out, index=False)
    print(f"Saved {out}")
    display(
        l1_df.sort_values(["l1", "Model"])
             .style.format({
                 "wer":           "{:.4f}",
                 "wer_base":      "{:.4f}",
                 "wer_delta_pct": "{:+.1f}%",
                 "per":           lambda v: f"{v:.4f}" if pd.notna(v) else "—",
             })
             .set_caption(f"{split.capitalize()} — Per-L1 WER")
    )


Saved results/model_perf_comparison/comparison_scripted_by_l1.csv


,l1,num_utts,wer,per,Model,wer_base,wer_delta_pct
12,Arabic,974,15.9964,14.5452,CTC Aux,0.2387,+6602.1%
6,Arabic,974,0.1125,0.0735,Naive LoRA FT,0.2387,-52.9%
0,Arabic,974,0.2387,0.1374,Zero-shot,0.2387,+0.0%
13,Chinese,1130,18.6869,16.6038,CTC Aux,0.1916,+9654.9%
7,Chinese,1130,0.0987,0.0582,Naive LoRA FT,0.1916,-48.5%
1,Chinese,1130,0.1916,0.1055,Zero-shot,0.1916,+0.0%
14,Hindi,1132,18.0533,16.7556,CTC Aux,0.0653,+27535.0%
8,Hindi,1132,0.0094,0.0051,Naive LoRA FT,0.0653,-85.5%
2,Hindi,1132,0.0653,0.0247,Zero-shot,0.0653,+0.0%
15,Korean,1131,18.9424,17.5941,CTC Aux,0.0810,+23284.8%


Saved results/model_perf_comparison/comparison_spontaneous_by_l1.csv


,l1,num_utts,wer,per,Model,wer_base,wer_delta_pct
12,Arabic,3,0.5233,0.4957,CTC Aux,0.5337,-1.9%
6,Arabic,3,0.5259,0.4985,Naive LoRA FT,0.5337,-1.5%
0,Arabic,3,0.5337,0.5104,Zero-shot,0.5337,+0.0%
13,Chinese,4,0.6259,0.5905,CTC Aux,0.6423,-2.6%
7,Chinese,4,0.6241,0.5899,Naive LoRA FT,0.6423,-2.8%
1,Chinese,4,0.6423,0.6078,Zero-shot,0.6423,+0.0%
14,Hindi,3,0.4428,0.3609,CTC Aux,0.4340,+2.0%
8,Hindi,3,0.4487,0.3621,Naive LoRA FT,0.4340,+3.4%
2,Hindi,3,0.4340,0.3567,Zero-shot,0.4340,+0.0%
15,Korean,4,0.7937,0.7457,CTC Aux,0.8089,-1.9%


 ---

 # Part 3 — Scripted vs Spontaneous Gap (zero-shot)



 Different corpora → compared at L1 level, not speaker level.

In [8]:
if available("baseline", "scripted") and available("baseline", "spontaneous"):
    s_g  = grouped_wer(results["baseline"]["scripted"],    "l1").rename(columns={"wer": "WER_scripted"})
    sp_g = grouped_wer(results["baseline"]["spontaneous"], "l1").rename(columns={"wer": "WER_spontaneous"})
    gap  = s_g[["l1", "WER_scripted"]].merge(
               sp_g[["l1", "WER_spontaneous"]], on="l1", how="inner")
    gap["gap"] = gap["WER_spontaneous"] - gap["WER_scripted"]

    l1s = gap["l1"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(name="Scripted",    x=l1s, y=gap["WER_scripted"].tolist()))
    fig.add_trace(go.Bar(name="Spontaneous", x=l1s, y=gap["WER_spontaneous"].tolist()))
    fig.update_layout(
        title   = "Zero-shot WER — Scripted vs Spontaneous by L1",
        barmode = "group",
        yaxis   = dict(title="WER", tickformat=".0%"),
        legend  = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
    )
    fig.show()
    display(
        gap.style
           .format({c: "{:.4f}" for c in ["WER_scripted", "WER_spontaneous", "gap"]})
           .set_caption("Scripted vs Spontaneous WER gap (zero-shot)")
    )


,l1,WER_scripted,WER_spontaneous,gap
0,Arabic,0.2387,0.5337,0.2950
1,Chinese,0.1916,0.6423,0.4508
2,Hindi,0.0653,0.4340,0.3687
3,Korean,0.0810,0.8089,0.7279
4,Spanish,0.2284,0.5957,0.3674
5,Vietnamese,0.3123,0.5510,0.2387


 ---

 # Part 4 — Utterance-level Analysis

In [9]:
# ── UTT WER distributions ─────────────────────────────────────────────────────
fig = go.Figure()
for key, label in MODEL_KEYS.items():
    for split in SPLITS:
        if not available(key, split):
            continue
        fig.add_trace(go.Histogram(
            x       = results[key][split]["utt_wer"],
            name    = f"{label} / {split}",
            opacity = 0.5,
            nbinsx  = 40,
        ))
fig.update_layout(
    title    = "Utterance WER Distribution — All Conditions",
    barmode  = "overlay",
    xaxis    = dict(title="Utterance WER"),
    yaxis    = dict(title="Count"),
    legend   = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
)
fig.show()


In [10]:
# ── Top-10 worst utterances — scripted zero-shot, cross-model comparison ──────
split = "scripted"
if available("baseline", split):
    base_df = results["baseline"][split]
    idx     = base_df.nlargest(10, "utt_wer").index
    worst   = base_df.loc[idx, [
        "utterance_id", "speaker", "l1",
        "reference_norm", "prediction_norm", "utt_wer", "utt_per",
    ]].rename(columns={
        "prediction_norm": "pred_baseline",
        "utt_wer":         "wer_baseline",
        "utt_per":         "per_baseline",
    })

    for key, label in MODEL_KEYS.items():
        if key == "baseline" or not available(key, split):
            continue
        other = results[key][split]
        worst[f"pred_{key}"] = other.loc[idx, "prediction_norm"].values
        worst[f"wer_{key}"]  = other.loc[idx, "utt_wer"].values
        if "utt_per" in other.columns:
            worst[f"per_{key}"] = other.loc[idx, "utt_per"].values

    fmt_cols = {c: "{:.3f}" for c in worst.columns if c.startswith(("wer_", "per_"))}
    display(
        worst.style
             .format(fmt_cols)
             .set_caption("Top-10 Worst Utterances — Zero-shot / Scripted")
    )


,utterance_id,speaker,l1,reference_norm,pred_baseline,wer_baseline,per_baseline,pred_baseline_lora,wer_baseline_lora,per_baseline_lora,pred_ctc_aux,wer_ctc_aux,per_ctc_aux
3422,arctic_a0155,HQTV,Vietnamese,wont you draw up gentlemen,one youve rarred and the other youve chandlemen,1.600,1.050,wont you draw up gentlemen,0.000,0.000,wont you draw up gentlemen,0.000,0.000
4179,arctic_b0319,HQTV,Vietnamese,daylight was tired profoundly tired,they lied to a tire a foully tire,1.600,0.480,they liked what tide roughly tide,1.200,0.440,they lied what tie roughly tied them up and said to me oh my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my my,88.600,35.320
1510,arctic_a0381,EBVS,Spanish,my names ferguson,my name is swargu sam,1.333,0.615,my name is swargoon,1.000,0.462,my name is ferguson,0.667,0.077
4780,arctic_a0381,SKA,Arabic,my names ferguson,my name is faye gasson,1.333,0.231,my name is gregson,1.000,0.385,my name is gregson,1.000,0.385
3577,arctic_a0310,HQTV,Vietnamese,massage under tension was the cryptic reply,much of the charges and the tensions were a critical reply,1.286,0.625,much much charge and attention was a great reply,1.000,0.438,much much audacious under tension was a cryptic reply and i thought that i was going to go back home and take a look at the book and read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i was going to read it out loud and i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i thought that i th